In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch, json, os, gc
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

LABEL_NAMES = {0:"Yardım Talebi",1:"Kayıp Bildirimi",2:"Altyapı Hasarı",
               3:"Bağış/Koordinasyon",4:"Diğer/İlgisiz"}
test_df = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/processed/test.csv')

def make_instruction(tweet):
    return ("Sen bir deprem acil durum sınıflandırma asistanısın. "
            "Verilen tweeti aşağıdaki kategorilerden birine koy. "
            "SADECE kategori adını yaz, başka hiçbir şey yazma.\n\n"
            f"Tweet: {tweet}\n\n"
            "Kategoriler:\n- Yardım Talebi\n- Kayıp Bildirimi\n- Altyapı Hasarı\n"
            "- Bağış/Koordinasyon\n- Diğer/İlgisiz")

def parse_label(text):
    t = text.lower()
    if "yardım" in t or "yardim" in t:                      return 0
    if "kayıp" in t or "kayip" in t:                        return 1
    if "altyapı" in t or "altyapi" in t:                    return 2
    if "bağış" in t or "bagis" in t or "koordinasyon" in t: return 3
    if "diğer" in t or "diger" in t or "ilgisiz" in t:      return 4
    return -1

MODELS = [
    ("smollm2", "SmolLM2-360M-Instruct", "HuggingFaceTB/SmolLM2-360M-Instruct"),
    ("qwen25",  "Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"),
]
out_dir = '/content/drive/MyDrive/bdm_proje/results/zero_shot'
os.makedirs(out_dir, exist_ok=True)

for KEY, NAME, MID in MODELS:
    print(f"\n{'='*55}\n{NAME} — ZERO-SHOT\n{'='*55}")
    tok = AutoTokenizer.from_pretrained(MID)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(MID, device_map="auto", dtype=torch.float16)
    mdl.eval()
    preds, trues, fails = [], [], 0
    for _, row in test_df.iterrows():
        msgs = [{"role":"user","content":make_instruction(row['tweet'])}]
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(prompt, return_tensors="pt").to(mdl.device)
        n = inp['input_ids'].shape[1]
        with torch.no_grad():
            out = mdl.generate(**inp, max_new_tokens=12, do_sample=False, pad_token_id=tok.pad_token_id)
        gen = tok.decode(out[0][n:], skip_special_tokens=True).strip()
        pid = parse_label(gen)
        if pid == -1: fails += 1
        preds.append(pid if pid!=-1 else 4); trues.append(int(row['label_id']))
    acc = accuracy_score(trues, preds)
    mf1 = f1_score(trues, preds, average='macro', zero_division=0)
    wf1 = f1_score(trues, preds, average='weighted', zero_division=0)
    print(f"Accuracy    : {acc:.4f} ({acc*100:.1f}%)")
    print(f"Macro-F1    : {mf1:.4f}\nWeighted-F1 : {wf1:.4f}")
    print(f"Parse fail  : {fails}/{len(test_df)}")
    sonuc = {"model":NAME,"yontem":"Zero-shot","test_size":int(len(test_df)),
             "accuracy":round(float(acc),4),"f1_macro":round(float(mf1),4),
             "f1_weighted":round(float(wf1),4),"parse_basarisiz":int(fails),
             "confusion_matrix":confusion_matrix(trues,preds,labels=[0,1,2,3,4]).tolist()}
    with open(os.path.join(out_dir,f'{KEY}_zero_shot_sonuc.json'),'w',encoding='utf-8') as f:
        json.dump(sonuc, f, ensure_ascii=False, indent=2)
    print(f"✅ Kaydedildi → {KEY}_zero_shot_sonuc.json")
    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

print("\n🎉 İki model zero-shot tamam.")


SmolLM2-360M-Instruct — ZERO-SHOT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Accuracy    : 0.4000 (40.0%)
Macro-F1    : 0.3906
Weighted-F1 : 0.3726
Parse fail  : 0/80
✅ Kaydedildi → qwen25_zero_shot_sonuc.json

🎉 İki model zero-shot tamam.
